## CS310 Natural Language Processing
## Lab 5: Byte-Pair Encoding (BPE) Tokenization

In this lab, we will practice the Byte-Pair Encoding (BPE) tokenization algorithm. The code for this lab is adopted from the [HuggingFace online course](https://huggingface.co/learn/llm-course/chapter6/1).

Install `transformers` before getting started.
```bash
pip install transformers
```

### T0. BPE Algorithm Procedure

In BPE, raw inputs are tokenized by applying the following steps:

1. Normalization `*`
2. Pre-tokenization `*`
3. Splitting the words into individual characters
4. Applying the merge rules learned in order on those splits

`*`: not covered in this lab.

---

BPE training starts by computing the unique set of words used in the corpus (after the normalization and pre-tokenization steps are completed), then building the vocabulary by taking all the symbols used. 

As a very simple example:

In [1]:
"hug", "pug", "pun", "bun", "hugs"

('hug', 'pug', 'pun', 'bun', 'hugs')

The base vocabulary will then be `["b", "g", "h", "n", "p", "s", "u"]`

Based on this vocabulary, we add new tokens until a certain vocabulary size is reached by the processing called `merging`:

- Merge two elements of the existing vocabulary together into a new one
- At the beginning, it will create tokens with `2` characters, and then, longer and longer subwords.
- At any step, the BPE algorithm will search for the `most frequent` pair of tokens, which is the one that will be merged.

Assume the words have following frequencies:

In [2]:
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)

(('hug', 10), ('pug', 5), ('pun', 12), ('bun', 4), ('hugs', 5))

Start by `splitting` each word into characters, so that each word is now a list of tokens:

In [3]:
("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)

(('hug', 10), ('pug', 5), ('pun', 12), ('bun', 4), ('hugs', 5))

The most frequence pair is `("u", "g")`, which appears 10 (in 'hug') + 5 (in 'pug') + 5 (in 'hugs') = 20 times

Thus, we merge it: `("u", "g") -> "ug"`, which results in a new vocabulary and corpus:

```Python
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug"]
Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)
```

---

Now, the most frequent pair at this stage is ("u", "n")

However, present 16 times in the corpus, so the second merge rule learned is `("u", "n") -> "un"`, leading to:

```Python
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un"]
Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("h" "ug" "s", 5)
```
---

Now the most frequent pair is ("h", "ug"), so we learn the merge rule ("h", "ug") -> "hug", which gives us our first three-letter token. After the merge, the corpus looks like this:

```Python
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un", "hug"]
Corpus: ("hug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("hug" "s", 5)
```
---

### T1. Implementing BPE 

Now we will go ahead and implement the BPE algorithm.

First we need a simple corpus:

In [4]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

Next, we need to `pre-tokenize` that corpus into words, and we will use the gpt2 tokenizer for the pre-tokenization:

In [6]:
import os
MODEL_HOME = os.environ.get("MODEL_HOME", None)
if MODEL_HOME is None:
    MODEL_HOME = os.path.join(os.environ.get('HOME'), 'models')
print(MODEL_HOME)


if MODEL_HOME:
    gpt2_path = os.path.join(MODEL_HOME, "gpt2")
else:
    gpt2_path = "gpt2"
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(gpt2_path)
print(f'tokenizer loaded from {gpt2_path}')

/Users/xy/models
tokenizer loaded from /Users/xy/models/gpt2


Compute the frequencies of each word in the corpus:

In [7]:
from collections import defaultdict

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)

defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})


The expected output is as follows:
```Python
defaultdict(int, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1,
    'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1,
    'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1,
    'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})
```
---

The next step is to compute the base vocabulary, formed by all the characters used in the corpus:

In [18]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(alphabet)

[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


In the obtained alphabet, we can find a special character `'Ġ'`. Its purpose is to avoid complexity in space. 

We also add the special tokens used by the GPT-2 model at the beginning of that vocabulary, `"<|endoftext|>"`:

In [9]:
vocab = ["<|endoftext|>"] + alphabet.copy()
print(vocab)

['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


---
We now need to split each word into individual characters, to be able to start training:

In [25]:
splits = {word: [c for c in word] for word in word_freqs.keys()}

# A helper function to reset the splits
def reset_splits():
    global splits
    splits = {word: [c for c in word] for word in word_freqs.keys()}

Let's take a look at `splits`:

In [ ]:
for i, (w, c_list) in enumerate(splits.items()):
    print(f"{w}: {c_list}")
    if i >= 5:
        break

# You expect to see the following output:
# This: ['T', 'h', 'i', 's']
# is: ['Ġi', 's']
# the: ['Ġt', 'h', 'e']
# Hugging: ['ĠH', 'u', 'g', 'g', 'i', 'n', 'g']
# Face: ['ĠF', 'a', 'c', 'e']
# Course: ['ĠC', 'o', 'u', 'r', 's', 'e']

This: ['T', 'h', 'i', 's']
Ġis: ['Ġ', 'i', 's']
Ġthe: ['Ġ', 't', 'h', 'e']
ĠHugging: ['Ġ', 'H', 'u', 'g', 'g', 'i', 'n', 'g']
ĠFace: ['Ġ', 'F', 'a', 'c', 'e']
ĠCourse: ['Ġ', 'C', 'o', 'u', 'r', 's', 'e']


---

Now that we are ready for training.

let’s write a function that computes the frequency of each token `pair`:

*Hint*: 
- The returned `pair_freqs` is a dictionary whose keys are token pairs (as tuples), and values are the frequencies. For instance,
`pair_freqs[('T', 'h')] = 3`
- For each `split` list correponding to a word, iterate through each token pair and get its frequency.

In [ ]:
def compute_pair_freqs(word_freqs: dict, splits: dict):
    """
    Return: pair_freqs, a dictionary of the frequency of each token pair. Its keys are tuples of two tokens, and its values are the frequencies.
    """
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1: # if it is a single character, skip
            continue

        ### START YOUR CODE ###
        # Hint: Use a for loop to go through the split list
        pass
        ### END YOUR CODE ###

    return pair_freqs

Let's look at part of the returned pair frequencies:

In [ ]:
# Test
pair_freqs = compute_pair_freqs(word_freqs, splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

# You expect to see the following output:
# ('T', 'h'): 3
# ('h', 'i'): 3
# ('i', 's'): 5
# ('Ġ', 'i'): 2
# ('Ġ', 't'): 7
# ('t', 'h'): 3

('T', 'h'): 3
('h', 'i'): 3
('i', 's'): 5
('Ġ', 'i'): 2
('Ġ', 't'): 7
('t', 'h'): 3


Now, finding the `most frequent pair` only takes a quick loop:

In [ ]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    ### START YOUR CODE ###
    # Find the most frequent pair and the frequency
    pass
    ### END YOUR CODE ###

# Test
print(best_pair, max_freq)
# You expect to see the following output:
# ('Ġ', 't') 7

('Ġ', 't') 7


So the first merge to learn is `('Ġ', 't') -> 'Ġt'`, and we add `'Ġt'` to the vocabulary:

In [22]:
# merges = {}
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")

To continue, we need to apply the merge in our `splits` dictionary. The merging operation is to update the values of `splits`.

After merging two characters, the original two characters should be removed from `splits`.

This is done in the `merge_pair` function:

*Hint*:
- The argument `(a, b)` defines the pair to be merge. For instance, if `a='Ġ'` and `b='t'`, and the list before merging is `split = ['Ġ', 't', 'r', 'a', 'i', 'n', 'e', 'd']`, then the merged list should be `new_list = ['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']`.

In [ ]:
def merge_pair(a, b, word_freqs: dict, splits: dict):
    """
    Args:
        (a, b) defines the pair to be merged.
    Returns:
        splits: the splits of each word in the corpus after merging the pair
    """
    for word in word_freqs: # go through each word in word_freqs
        split = splits[word] # get the split of the word
        if len(split) == 1: # if the word is a single character, skip
            continue

        ### START YOUR CODE ###
        # Hint: Use a for loop to go through the split list and find the pair (a, b)
        # If found, replace the pair with the merged token (a+b)
        # Otherwise, keep the list unchanged
        pass
        ### END YOUR CODE ###

    return splits

Let's check the result of the first merge:

In [26]:
reset_splits() # reset the splits

print('before merging:')
print(splits["Ġtrained"])

splits = merge_pair("Ġ", "t", word_freqs, splits)

print('after merging:')
print(splits["Ġtrained"])

before merging:
['Ġ', 't', 'r', 'a', 'i', 'n', 'e', 'd']
after merging:
['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']


---

Now we have everything we need to loop until we have learned all the merges we want. 

**Let’s aim for a vocab size of 50:**

In [ ]:
# Reset vocabulary to base (alphabet + special token) before training merges
vocab = ["<|endoftext|>"] + alphabet.copy()
merges = {}

vocab_size = 50
while len(vocab) < vocab_size:
    ### START YOUR CODE ###
    # Call compute_pair_freqs to get the frequency of each token pair
    pair_freqs = None

    # Find the most frequent pair
    best_pair = ""
    max_freq = None

    # Call merge_pair to merge the best pair        
    splits = None
    ### END YOUR CODE ###

    # Record the merge and update the vocabulary
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

As a result, we’ve learned 19 merge rules (the initial vocabulary had a size of 31 — 30 characters in the alphabet, plus the special token):

In [ ]:
# Test
print(merges)
print(len(merges))

# You expect to see the following output:
# {('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni', ('Ġtokeni', 'z'): 'Ġtokeniz'}
# 19

{('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni', ('Ġtokeni', 'z'): 'Ġtokeniz'}
19


And the vocabulary is composed of the special token, the initial alphabet, and all the results of the merges:

In [ ]:
# Test
print(vocab)
print(len(vocab))

# You expect to see the following output:
# ['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis', 'Ġth', 'Ġthe', 'in', 'Ġab', 'Ġtokeni', 'Ġtokeniz']
# 50

['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis', 'Ġth', 'Ġthe', 'in', 'Ġab', 'Ġtokeni', 'Ġtokeniz']
50


It is exactly the intended vocabulary size of 50.